Убираем все warnings:

In [ ]:
import warnings
warnings.filterwarnings('default') # ignore

Подключаем google disk для доступа к датасету и чекпоинтам:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Устанавливаем зависимости:

In [ ]:
!pip install datasets pyannote.metrics pyannote.audio huggingface_hub optuna onnxruntime onnxruntime-gpu

Получаем доступ к Hugging Face Hub. Для этого нужно настроить переменную окружения HF_TOKEN в Google Colab. Библиотека huggingface_hub сама найдет в окружении эту переменную и будет использовать ее значение.

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import os

try:
    print("Logging in HuggingFace...")
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    print("Token has downloaded from Colab's secrets!")
    login(token=hf_token)
    print("Login successfully!")
except Exception as e:
    print(f"Error: {e}")
    hf_token = None

Переводим исполнение на GPU, если это возможно. Данное действие позволит значительно ускорить подбор гиперпараметров в отличие от исполнения на CPU.

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Selected device: {DEVICE}")

Подкачиваем файлы из выборки development. Для этого нужно указать путь к database.yml конфигу, где прописаны пути ко всем выборкам.

In [ ]:
from pyannote.database import get_protocol, registry

# Заменить путь к файлу database.yml на корректный, если это требуется
config_path = '/content/drive/MyDrive/AMI-diarization-setup/pyannote/database.yml'
registry.load_database(config_path)
protocol = get_protocol('AMI.SpeakerDiarization.word_and_vocalsounds')
files = list(protocol.development())
print(f"Loaded {len(files)} files from development subset")

Подкачиваем дообученную модель сегментации из лучшего checkpoint.

In [ ]:
from pyannote.audio import Model

# Заменить путь к лучшему checkpoint, если это требуется
SEG_CKPT = "/content/drive/MyDrive/pyannote_finetuning/ami_segmentation_v1/fixed_version/checkpoints/best-epoch=09-step=5329.ckpt"
segmentation_model = Model.from_pretrained(SEG_CKPT)
print("Segmentation model loaded directly from checkpoint")

Выстраиваем пайплайн диаризации из частей:
1. Дообученная модель сегментации
2. Модель для извлечения эмбеддингов
3. Алгоритм кластеризации

In [ ]:
from pyannote.audio import Pipeline
from pyannote.audio.utils.powerset import Powerset
from pyannote.audio.pipelines import SpeakerDiarization
from pyannote.audio.pipelines.clustering import AgglomerativeClustering

pretrained_pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1")

base_pipeline = SpeakerDiarization(
    segmentation=segmentation_model,
    embedding=pretrained_pipeline.embedding,
    embedding_exclude_overlap=pretrained_pipeline.embedding_exclude_overlap,
    clustering="AgglomerativeClustering"
)

Заполняем pyannote_data данными о файлах из development для дальнейшегго использования при подборе гиперпараметров.

In [ ]:
CORPUS_ROOT = "/content/drive/MyDrive/AMI-diarization-setup/pyannote/"

pyannote_data = []
for file in files:
    uri = file['uri']
    audio_path = os.path.join(CORPUS_ROOT, f"amicorpus/{uri}/audio/{uri}.Mix-Headset.wav")

    if not os.path.exists(audio_path):
        print(f"Warning: {audio_path} not found, skipping")
        continue

    pyannote_data.append({
        "uri": uri,
        "audio": audio_path,
        "annotation": file['annotation'],
        "subset": None
    })

print(f"Prepared {len(pyannote_data)} files")

Настраиваем ячейку для подбора гиперпараметров с помощью библиотеки Optuna, которая является стандартом для оптимизации гиперпараметров в Pyannote. Ключевые моменты:
1. 60 попыток подбора гиперпараметров
2. Подбор min_duration_off для модели сегментации и min_cluster_size, threshold для кластеризации
3. Созранение состояний в базу данных SQLite для возможности восстанноваления процесса после сбоя или остановки Google Colab
4. Будет минимизироваться ошибка DER

In [ ]:
import optuna
import optuna.visualization as vis
from optuna.storages import RDBStorage
import time
from pyannote.metrics.diarization import DiarizationErrorRate
from pyannote.audio import Pipeline

NUM_TRIALS = 60
DB_PATH = "/content/drive/MyDrive/optuna_study.sqlite3"

storage = RDBStorage(f"sqlite:///{DB_PATH}")
study_name = "speaker_diarization_study"

def objective(trial):
    print(f"=================================== Trial: {trial.number} ===================================")
    params = {
        "segmentation": {
            "min_duration_off": trial.suggest_float("min_duration_off", 0.0, 1.0),
        },
        "clustering": {
            "method": 'centroid',
            "min_cluster_size": trial.suggest_int("min_cluster_size", 2, 20),
            "threshold": trial.suggest_float("clustering_threshold", 0.0, 1.0),
        }
    }

    base_pipeline.instantiate(params)
    base_pipeline.to(torch.device(DEVICE))
    metric = DiarizationErrorRate()
    for i, file in enumerate(pyannote_data):
        print(f"Processing file {i+1}/{len(pyannote_data)}: {file['uri']}")
        start = time.time()
        hypothesis = base_pipeline(file)
        hypothesis_diarization = hypothesis.speaker_diarization
        end = time.time()
        print(f"  Time: {end - start}s.")
        metric(file["annotation"], hypothesis_diarization)

    der = abs(metric)
    print(f"Trial {trial.number} DER = {der:.2%}")
    return der

In [ ]:
try:
    study = optuna.load_study(study_name=study_name, storage=storage)
    print(f"Loaded existing study with {len(study.trials)} trials already completed.")
except KeyError:
    study = optuna.create_study(
        study_name=study_name,
        storage=storage,
        load_if_exists=True,
        direction="minimize",
    )
    print("Created new study.")

Визуализация истории оптимизации:

In [ ]:
vis.plot_optimization_history(study).show()
vis.plot_param_importances(study).show()
print(f"The best params: {study.best_params}")
print(f"The best DER: {study.best_value:.2%}")

Начинаем подбор:

In [ ]:
study.optimize(objective, n_trials=NUM_TRIALS)